Importing libraries.

In [55]:
import pandas as pd
import numpy as np
import requests
import re
import os
import time
from lxml import html
from bs4 import BeautifulSoup
from tqdm import tqdm
import matplotlib.pyplot as plt

import selenium
from webdriver_manager.chrome import ChromeDriverManager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC 
from selenium.webdriver.common.by import By 


Let's define the page we will be scraping.

In [2]:
url = "https://nbp.pl/kategoria/aktualnosci/"

Scraping as a static page.

In [ ]:
response = requests.get(url)
print(response.text)

As we can see, website redirects us to its antirobot script, so it is not possible to access the content with simple requests.get. Maybe we can adjust headers of request and try once again?

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
}
response = requests.get(url, headers = headers)

print(response.text)

It seems that simple requests library does not set us up for success in this matter, thus a sway towards selenium might be of help here.

Setting up.

In [3]:

#chromepath = ChromeDriverManager().install()
#service = Service(executable_path = chromepath)
options = webdriver.ChromeOptions()
#driver = webdriver.Chrome(service = service, options = options)
driver = webdriver.Chrome(options = options)
driver.maximize_window()
driver.get(url)

Let's display the site's content and check if it works now. After switching to display the cell's output as a scrollable element and further examination, it is apparent that selenium has succeeded. 

In [ ]:
time.sleep(5+np.random.gamma(1,3))
print(driver.page_source)


Now let's deal with accepting cookies.

In [4]:
cookies_button_xpath = '''//*[@data-testid="actionButton-accept"]'''
WebDriverWait(driver, 10).until(EC.visibility_of_element_located((By.XPATH, cookies_button_xpath))) 
time.sleep(np.random.gamma(1, 3))
content = driver.find_element("xpath", cookies_button_xpath)
content.click()

Now that we finally reached site, the idea is as follows: let's find all &lt;a hrefs&gt; on the site in the &lt;body&gt; part, then let's proceed to filter out links that do not direct us to the news but some other subpages. Now, having found these parts of html code, let's save link, article name and publication date to a respective list and after that create Pandas DataFrame. 

One of the challenges is that the webpage of news' archive of nbp.pl displays only 10 most recent articles, so throughout the scraping process and upon checking the content of &lt;article class="entry"&gt; it is required that after checking the scraper proceeds to the next page of search results. Alas, next page of results isn't located on another website such as "www.nbp.pl/kategoria//aktualnosci/1", "www.nbp.pl/kategoria//aktualnosci/2" etc., but on the same page and the site only allows to click the button to execute a script and stay on the same page.

In [5]:
# collected links, titles, dates
hrefs = []
titles = []
dates = []
count = 0

number_of_pages_to_scrape = 129

for page in range(1,number_of_pages_to_scrape+1):
    print(f"Scraping page {page}/{number_of_pages_to_scrape}")
    
    #refinding elements on current results' page
    tags = driver.find_elements(By.CLASS_NAME, "entry")
    
    #extracting the data from current batch
    for tag in tags:
        try:
            
            #extracting info from one element of class selenium.webdriver.remote.webelement
            title_link_element = tag.find_element(By.CSS_SELECTOR, "h2.entry-title a")
            link = title_link_element.get_attribute("href")
            title = title_link_element.text.strip()
            date_element = tag.find_element(By.CSS_SELECTOR, "time.entry-date")
            date = date_element.text.strip() 
            
            #adding to respective lists
            hrefs.append(link)
            titles.append(title)
            dates.append(date)
            
            count += 1
            #print(f"{count}: {title}")
            
        except Exception as e:
            print(f"Skipping an element due to formatting error: {e}")

    # 3. Click the specific page number button to load the next set
    next_page_num = page + 1 # Since 'page' starts at 0, the first button we click is '2'
    
    # If we just scraped the final page, we don't need to click anything
    if next_page_num > number_of_pages_to_scrape:
        break

    print(f"Finished scraping page {page + 1}. Clicking button for page {next_page_num}...")
    
    try:
        # Find the <a> tag with class 'page-link' that contains the text of our target number
        next_button_xpath = f"//a[contains(@class, 'page-link') and text()='{next_page_num}']"
        next_button = driver.find_element(By.XPATH, next_button_xpath)
        
        # PRO-TIP: Pagination buttons are usually at the bottom. 
        # Scrolling to it prevents errors where a floating footer blocks the click.
        driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
        time.sleep(1) # Brief pause to let the scroll finish
        
        # Click it!
        next_button.click()
        
        # Wait for the old elements to disappear and new ones to load
        time.sleep(3 + np.random.gamma(1, 3))
        
    except Exception as e:
        print(f"Could not find or click the button for page {next_page_num}. Reached the end of results or got blocked.")
        break # Exit the outer loop if it fails

print(f"\nExtraction complete! Successfully collected {len(titles)} articles.")

Scraping page 1/129
Finished scraping page 2. Clicking button for page 2...
Scraping page 2/129
Finished scraping page 3. Clicking button for page 3...
Scraping page 3/129
Finished scraping page 4. Clicking button for page 4...
Scraping page 4/129
Skipping an element due to formatting error: Message: no such element: Unable to locate element: {"method":"css selector","selector":"time.entry-date"}
  (Session info: chrome=147.0.7727.138); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
0   chromedriver                        0x0000000103295ebc cxxbridge1$str$ptr + 3213376
1   chromedriver                        0x000000010328de8c cxxbridge1$str$ptr + 3180560
2   chromedriver                        0x0000000102d538d4 _RNvCs10ygTOo3JCa_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75096
3   chromedriver                        0x0000000102d9b8b4 _RNvCs10ygTOo3JCa_7___rustc35___rust_

In [6]:
df = pd.DataFrame({
    "titles": titles,
    "dates" : dates,
    "hrefs" : hrefs
})
df.to_csv("scraped_df.csv")

In [50]:
df = pd.read_csv("scraped_df.csv")
df = df.drop(columns = "Unnamed: 0")
def isRPP(text):
    pattern = re.compile(r"RPP|Rad\w*\s+Polity\w*\s+Pieniężn\w*", re.IGNORECASE)
    if pattern.search(text):
        return True

    return False

In [51]:
df["titles"].map(isRPP)
df_rpp = df[df["titles"].map(isRPP)]
df

,titles,dates,hrefs
0,Wyniki przetargu sprzedaży bonów pieniężnych N...,"8 maja, 2026",https://nbp.pl/wyniki-przetargu-sprzedazy-bono...
1,Oświadczenie rzecznika prasowego NBP,"7 maja, 2026",https://nbp.pl/oswiadczenie-rzecznika-prasoweg...
2,Aktywa rezerwowe Polski w kwietniu 2026 r.,"7 maja, 2026",https://nbp.pl/aktywa-rezerwowe-polski-w-kwiet...
3,"Ocena bieżącej sytuacji ekonomicznej w Polsce,...","7 maja, 2026",https://nbp.pl/ocena-biezacej-sytuacji-ekonomi...
4,Komunikat prasowy z posiedzenia Rady Polityki ...,"6 maja, 2026",https://nbp.pl/rpp-06-05-2026/
...,...,...,...
1269,Nowe kryteria wyboru przez NBP kontrahentów do...,"8 listopada, 2019",https://nbp.pl/nowe-kryteria-wyboru-przez-nbp-...
1270,Zmiana listy Uczestników Fixingu Stawki Refere...,"6 maja, 2019",https://nbp.pl/zmiana-listy-uczestnikow-fixing...
1271,Rozszerzenie grona Uczestników Fixingu Stawki ...,"11 kwietnia, 2019",https://nbp.pl/rozszerzenie-grona-uczestnikow-...
1272,NBP nowym Organizatorem Fixingu Stawki Referen...,"1 grudnia, 2017",https://nbp.pl/nbp-nowym-organizatorem-fixingu...


In [39]:
del df

In [52]:
months_dict = {
    "stycznia,": 1,
    "lutego,": 2,
    "marca,": 3,
    "kwietnia,": 4,
    "maja,": 5,
    "czerwca,": 6,
    "lipca,": 7,
    "sierpnia,": 8,
    "września,": 9,
    "października,": 10,
    "listopada,": 11,
    "grudnia,": 12
}
def parse_date(date_str):
    day, month_name, year = date_str.split()

    month = months_dict[month_name]

    return pd.Timestamp(
        year=int(year),
        month=month,
        day=int(day)
    )


In [53]:
df["dates"] = df["dates"].map(parse_date)


In [59]:
df_rpp

,titles,dates,hrefs
4,Komunikat prasowy z posiedzenia Rady Polityki ...,"6 maja, 2026",https://nbp.pl/rpp-06-05-2026/
5,Zmiana czerwcowego terminu posiedzenia RPP,"6 maja, 2026",https://nbp.pl/zmiana-czerwcowego-terminu-posi...
21,Komunikat prasowy z posiedzenia Rady Polityki ...,"9 kwietnia, 2026",https://nbp.pl/rpp-09-04-2026/
47,Komunikat prasowy z posiedzenia Rady Polityki ...,"4 marca, 2026",https://nbp.pl/rpp-04-03-2026/
62,Komunikat prasowy z posiedzenia Rady Polityki ...,"4 lutego, 2026",https://nbp.pl/rpp-04-02-2026/
...,...,...,...
1182,Komunikat prasowy z posiedzenia RPP w dniu 5 m...,"5 maja, 2021",https://nbp.pl/komunikat-prasowy-z-posiedzenia...
1202,Komunikat prasowy z posiedzenia RPP w dniu 7 k...,"7 kwietnia, 2021",https://nbp.pl/komunikat-prasowy-z-posiedzenia...
1226,Komunikat prasowy z posiedzenia RPP w dniu 3 m...,"3 marca, 2021",https://nbp.pl/komunikat-prasowy-z-posiedzenia...
1245,Komunikat prasowy z posiedzenia RPP w dniu 3 l...,"3 lutego, 2021",https://nbp.pl/komunikat-prasowy-z-posiedzenia...
